## Install and import the needed libraries
Pandas is needed to manipulate, analyze, and clean structured data.

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

## Download survey_data.csv and store it to df_original

In [ ]:
df_original = pd.read_csv('survey_data.csv')

In [ ]:
pd.set_option('display.max_columns',None) #to see all the columns of the dataset
df_original.head()

## Filter columns to only relavant columns for analysis

In [ ]:
#select the columns that we want for an analysis
relavant_columns = ['ResponseId','Age','Country','EdLevel',
                    'LanguageHaveWorkedWith','LanguageWantToWorkWith',
                    'DatabaseHaveWorkedWith','DatabaseWantToWorkWith',
                    'PlatformHaveWorkedWith','PlatformWantToWorkWith',
                    'WebframeHaveWorkedWith','WebframeWantToWorkWith']

df = df_original[relavant_columns].copy()

Check the size (dimensions) of the dataset: (rows,columns)

In [ ]:
df.shape

See the first 5 rows of the dataset

In [ ]:
df.head()

## Check for missing values

In [ ]:
missing_values = df.isnull().sum()

In [ ]:
df_missing = pd.DataFrame(missing_values.reset_index(name='missing_count'))
df_missing

There are many missing value in this dataset.

Let's see the percentage of missing values in each column.

In [ ]:
missing_values_percent = round(df.isnull().sum() / len(df) *100,2)

In [ ]:
df_missing_percent = pd.DataFrame(missing_values_percent.reset_index(name='percent'))
df_missing_percent

## Handling missing values

#### Country
Missing values in 'Country' likely indicate that respondents chose not to disclose their location, so they are replaced with "Unknown".

In [ ]:
df['Country'] = df['Country'].fillna('Unknown')

#### EdLevel (Educational Level)
Missing values in 'EdLevel' likely indicate that respondents chose not to disclose their education level, so they are replaced with "Unknown".

In [ ]:
df['EdLevel'] = df['EdLevel'].fillna('Unknown')

#### Others technology columns
Missing values likely indicate that respondents did not select any technology, so they are replaced with "None".

In [ ]:
tech_cols = ['LanguageHaveWorkedWith','LanguageWantToWorkWith',
             'DatabaseHaveWorkedWith','DatabaseWantToWorkWith',
             'PlatformHaveWorkedWith','PlatformWantToWorkWith',
             'WebframeHaveWorkedWith','WebframeWantToWorkWith']

df[tech_cols] = df[tech_cols].fillna('None')

#### Check for missing values again

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

## Clean data to make it easier to analyze and visualize

#### Age

In [ ]:
age_group = {'Under 18 years old': 'Under 18y',
            '18-24 years old': '18-24y',
            '25-34 years old': '25-34y',
            '35-44 years old': '35-44y',
            '45-54 years old': '45-54y',
            '55-64 years old': '55-64y',
            '65 years or older': '65y+'}

df['Age'] = df['Age'].replace(age_group)

df['Age'].value_counts()

#### Country

In [ ]:
long_name_countries = df['Country'][df['Country'].str.len() > 12].unique()
long_name_countries

In [ ]:
country_map = {
    'United Kingdom of Great Britain and Northern Ireland': 'UK',
    'United States of America': 'USA',
    'Republic of North Macedonia': 'North Macedonia',
    'United Republic of Tanzania': 'Tanzania',
    'Czech Republic': 'Czechia',
    'Russian Federation': 'Russia',
    'Venezuela, Bolivarian Republic of...': 'Venezuela',
    'Hong Kong (S.A.R.)': 'Hong Kong',
    'United Arab Emirates': 'UAE',
    'Iran, Islamic Republic of...': 'Iran',
    'Democratic Republic of the Congo': 'DR Congo',
    'Republic of Korea': 'South Korea',
    'Syrian Arab Republic': 'Syria',
    'Côte d\'Ivoire': 'Ivory Coast',   # <-- escaped apostrophe
    'Republic of Moldova': 'Moldova',
    'Congo, Republic of the...': 'Republic of the Congo',
    'Brunei Darussalam': 'Brunei'
}

df['Country'] = df['Country'].replace(country_map)

df['Country'].value_counts()

#### EdLevel (Education Level)

In [ ]:
education_map = {'Bachelor’s degree (B.A., B.S., B.Eng., etc.)': 'Bachelor’s degree',
                'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)': 'Master’s degree',
                'Some college/university study without earning a degree': 'Study wo degree',
                'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 'Secondary School',
                'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 'Professional degree',
                'Associate degree (A.A., A.S., etc.)': 'Associate degree',
                'Primary/elementary school': 'Primary school'}

df['EdLevel'] = df['EdLevel'].replace(education_map)

df['EdLevel'].value_counts()

### These columns below contain multiple answers from the respondents (Multiple-Values Columns)

These columns are LanguageHaveWorkedWith, LanguageWantToWorkWith, DatabaseHaveWorkedWith, DatabaseWantToWorkWith, PlatformHaveWorkedWith, PlatformWantToWorkWith, WebframeHaveWorkedWith, and WebframeWantToWorkWith 

We need to split each of them by ; 

#### Programming Language

In [ ]:
df_lang = df[['ResponseId', 'Age', 'Country', 'EdLevel',
              'LanguageHaveWorkedWith', 'LanguageWantToWorkWith']].copy()

df_lang['LanguageHaveWorkedWith'] = df_lang['LanguageHaveWorkedWith'].str.split(';')
df_lang = df_lang.explode('LanguageHaveWorkedWith')
df['LanguageHaveWorkedWith'] = df['LanguageHaveWorkedWith'].str.strip()

df_lang['LanguageWantToWorkWith'] = df_lang['LanguageWantToWorkWith'].str.split(';')
df_lang = df_lang.explode('LanguageWantToWorkWith')
df_lang['LanguageWantToWorkWith'] = df_lang['LanguageWantToWorkWith'].str.strip()

In [ ]:
df_lang.head()

#### Database

In [ ]:
df_db = df[['ResponseId','Age','Country','EdLevel',
            'DatabaseHaveWorkedWith','DatabaseWantToWorkWith']].copy()

df_db['DatabaseHaveWorkedWith'] = df_db['DatabaseHaveWorkedWith'].str.split(';')
df_db = df_db.explode('DatabaseHaveWorkedWith')
df_db['DatabaseHaveWorkedWith'] = df_db['DatabaseHaveWorkedWith'].str.strip()

df_db['DatabaseWantToWorkWith'] = df_db['DatabaseWantToWorkWith'].str.split(';')
df_db = df_db.explode('DatabaseWantToWorkWith')
df_db['DatabaseWantToWorkWith'] = df_db['DatabaseWantToWorkWith'].str.strip()

In [ ]:
df_db.head()

#### Platform

In [ ]:
df_pl = df[['ResponseId','Age','Country','EdLevel',
                  'PlatformHaveWorkedWith','PlatformWantToWorkWith']].copy()

df_pl['PlatformHaveWorkedWith'] = df_pl['PlatformHaveWorkedWith'].str.split(';')
df_pl = df_pl.explode('PlatformHaveWorkedWith')
df_pl['PlatformHaveWorkedWith'] = df_pl['PlatformHaveWorkedWith'].str.strip()

df_pl['PlatformWantToWorkWith'] = df_pl['PlatformWantToWorkWith'].str.split(';')
df_pl = df_pl.explode('PlatformWantToWorkWith')
df_pl['PlatformWantToWorkWith'] = df_pl['PlatformWantToWorkWith'].str.strip()

In [ ]:
df_pl.head()

#### Web Frame

In [ ]:
df_web = df[['ResponseId','Age','Country','EdLevel',
             'WebframeHaveWorkedWith','WebframeWantToWorkWith']].copy()

df_web['WebframeHaveWorkedWith'] = df_web['WebframeHaveWorkedWith'].str.split(';')
df_web = df_web.explode('WebframeHaveWorkedWith')
df_web['WebframeHaveWorkedWith'] = df_web['WebframeHaveWorkedWith'].str.strip()

df_web['WebframeWantToWorkWith'] = df_web['WebframeWantToWorkWith'].str.split(';')
df_web = df_web.explode('WebframeWantToWorkWith')
df_web['WebframeWantToWorkWith'] = df_web['WebframeWantToWorkWith'].str.strip()

In [ ]:
df_web.head()

## Check the cleaned data

In [ ]:
df.head()

## Export Files

In [ ]:
df.to_csv('survey_data_updated.csv', index=False)
df_lang.to_csv('languages.csv', index=False)
df_db.to_csv('databases.csv', index=False)
df_pl.to_csv('platforms.csv', index=False)
df_web.to_csv('webframes.csv', index=False)